# Elenchus — serving the Socratic tutor from Colab

Runs the team's fine-tuned model on Colab's free GPU and puts a public address in
front of it, so anyone's VS Code extension can reach it with nothing installed.

**Before you start:** Runtime → Change runtime type → **T4 GPU**.

Then run both cells. The second one prints an address and stops there, holding the
session open. Paste that address into `endpoint.txt` on GitHub and the whole team
is connected.

Colab disconnects after a while, and the address changes each time you restart.
That is exactly what `endpoint.txt` is for: edit one line, everyone follows.

In [ ]:
#@title 1. Set up (about two minutes)
import subprocess, sys, os, pathlib

# The model, the server and the web page all live in the public HF Space repo.
# The Space itself cannot run (no quota on that account), but it works perfectly
# well as somewhere to keep the files.
REPO = "https://huggingface.co/spaces/Alsalay/elenchus_poc"

if not pathlib.Path("elenchus").exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO, "elenchus"], check=True)
os.chdir("/content/elenchus")

# torch and transformers ship with Colab; peft is the one that does not.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft"], check=True)

# cloudflared gives a public https address with no account and no signup. It runs
# here on Google's machines, so any restriction on your own connection is irrelevant.
if not pathlib.Path("/usr/local/bin/cloudflared").exists():
    subprocess.run([
        "wget", "-q", "-O", "/usr/local/bin/cloudflared",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
    ], check=True)
    subprocess.run(["chmod", "+x", "/usr/local/bin/cloudflared"], check=True)

import torch
print("\nGPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — set Runtime to T4")
print("files:", sorted(p.name for p in pathlib.Path('.').iterdir() if not p.name.startswith('.')))

In [ ]:
#@title 2. Start the tutor and get its address
import subprocess, threading, queue, re, time, sys, os, urllib.request

PORT = 8008

server = subprocess.Popen(
    [sys.executable, "serve.py", "--host", "127.0.0.1", "--port", str(PORT)],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)

print("loading the model (downloads ~1 GB the first time)...")
ready = False
for _ in range(120):
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PORT}/v1/models", timeout=2).read()
        ready = True
        break
    except Exception:
        if server.poll() is not None:
            print("the server stopped. Its output:")
            print(server.stdout.read())
            raise SystemExit(1)
        time.sleep(3)
if not ready:
    raise SystemExit("the server did not come up in time")
print("model ready\n")

# cloudflared prints the address once, into its own output, so it has to be read
# as it goes rather than waited for.
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
found = queue.Queue()

def watch():
    for line in tunnel.stdout:
        m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line)
        if m:
            found.put(m.group(0))

threading.Thread(target=watch, daemon=True).start()
try:
    url = found.get(timeout=90)
except queue.Empty:
    raise SystemExit("cloudflared did not report an address")

print("=" * 68)
print("  THE TUTOR IS LIVE AT")
print()
print("     " + url + "/v1")
print()
print("  Put that line into endpoint.txt on GitHub:")
print("  https://github.com/H7Feez/elenchus_poc/edit/main/endpoint.txt")
print()
print("  Everyone's extension picks it up within five minutes.")
print("  Open " + url + " in a browser to try it without VS Code.")
print("=" * 68)
print("\nLeave this cell running. Replies appear below as people use it.\n")

try:
    for line in server.stdout:
        print(line, end="")
except KeyboardInterrupt:
    print("\nstopping")
finally:
    tunnel.terminate()
    server.terminate()